# Download Semua Data Harga Saham (Fixed)
Notebook ini digunakan untuk mengunduh semua data harga historis saham yang terdaftar di `List-Perusahaan/Perusahaan.csv` dan menyimpannya ke dalam folder `Dataset/` secara otomatis.

**FIX PENTING:** Versi sebelumnya pakai `pd.concat` + `drop_duplicates(keep='last')` untuk gabung data lama & baru. Karena `saham` (download baru dari yfinance) TIDAK punya kolom broker summary, dan ditaruh setelah `old` di concat, `keep='last'` selalu memenangkan baris baru yang kolom brokernya NaN — jadi SEMUA data broker yang udah discrape ke-wipe tiap kali notebook ini dijalankan ulang.

Fix di sini: update kolom HARGA saja lewat index tanggal, kolom broker (dan kolom lain di luar harga) sama sekali gak disentuh.

In [1]:
import yfinance as yf
import pandas as pd
import os
import time

KOLOM_HARGA = ["Open", "High", "Low", "Close", "Volume"]

def historical_price(Code_Saham):
    path_csv = f"./Dataset/Harga_Saham/{Code_Saham}.csv"

    # Buat folder Dataset otomatis di root jika belum ada
    os.makedirs(os.path.dirname(path_csv), exist_ok=True)

    # Download data
    saham = yf.download(Code_Saham, period="5d", interval="1d", progress=False)
    if saham.empty:
        raise ValueError("Data kosong atau ticker tidak ditemukan di yfinance")

    # Fix multi-index header
    saham.columns = [col[0] if isinstance(col, tuple) else col for col in saham.columns]

    # Jadikan Date kolom biasa lalu langsung dijadikan index string 'YYYY-MM-DD'
    saham.reset_index(inplace=True)
    saham["Date"] = pd.to_datetime(saham["Date"]).dt.strftime("%Y-%m-%d")
    saham = saham.set_index("Date")

    kolom_harga_ada = [c for c in KOLOM_HARGA if c in saham.columns]

    if os.path.exists(path_csv):
        old = pd.read_csv(path_csv)
        old["Date"] = pd.to_datetime(old["Date"]).dt.strftime("%Y-%m-%d")
        old = old.set_index("Date")

        # 1) Update HANYA kolom harga untuk tanggal yang udah ada di old.
        #    Kolom broker & kolom lain di old SAMA SEKALI GAK DISENTUH.
        tanggal_overlap = old.index.intersection(saham.index)
        for col in kolom_harga_ada:
            old.loc[tanggal_overlap, col] = saham.loc[tanggal_overlap, col]

        # 2) Tambahin tanggal baru yang belum ada di old (row baru, kolom
        #    broker otomatis NaN karena emang belum pernah discrape)
        tanggal_baru = saham.index.difference(old.index)
        if len(tanggal_baru) > 0:
            df = pd.concat([old, saham.loc[tanggal_baru]])
        else:
            df = old

        df = df.sort_index()
    else:
        df = saham.sort_index()

    # Simpan kembali ke CSV, Date balik jadi kolom biasa
    df = df.reset_index()
    df.to_csv(path_csv, index=False)
    return len(df)


In [2]:
# 1. Load list perusahaan
list_perusahaan_path = "./Dataset/List-Perusahaan/Perusahaan.csv"
if not os.path.exists(list_perusahaan_path):
    raise FileNotFoundError(f"File {list_perusahaan_path} tidak ditemukan! Pastikan Anda sudah menjalankan scraping list perusahaan terlebih dahulu.")

df_perusahaan = pd.read_csv(list_perusahaan_path)
tickers = df_perusahaan['Ticker_YF'].dropna().unique().tolist()
total_tickers = len(tickers)

print(f"Total saham yang akan didownload: {total_tickers}")


Total saham yang akan didownload: 915


In [3]:
# 2. Loop download dengan error handling
success_count = 0
failed_tickers = []

print("Memulai proses download...\n")

for i, ticker in enumerate(tickers, 1):
    try:
        # Beri jeda 0.5 detik agar tidak terkena rate limit dari API Yahoo Finance
        time.sleep(0.5)

        total_rows = historical_price(ticker)
        success_count += 1
        print(f"[{i}/{total_tickers}] {ticker}: BERHASIL ({total_rows} baris data)")

    except Exception as e:
        failed_tickers.append((ticker, str(e)))
        print(f"[{i}/{total_tickers}] {ticker}: GAGAL - {str(e)}")

print("\n=======================================")
print(f"Proses Selesai!")
print(f"Berhasil: {success_count}/{total_tickers}")
print(f"Gagal: {len(failed_tickers)}/{total_tickers}")
print("=======================================")

if failed_tickers:
    print("\nDaftar ticker yang gagal:")
    for t, err in failed_tickers:
        print(f"- {t}: {err}")


Memulai proses download...

[1/915] JPFA.JK: BERHASIL (4 baris data)
[2/915] CPIN.JK: BERHASIL (4 baris data)
[3/915] RLCO.JK: BERHASIL (4 baris data)
[4/915] WMUU.JK: BERHASIL (4 baris data)
[5/915] ASHA.JK: BERHASIL (4 baris data)
[6/915] CPRO.JK: BERHASIL (4 baris data)
[7/915] IKAN.JK: BERHASIL (4 baris data)
[8/915] AYAM.JK: BERHASIL (4 baris data)
[9/915] MAIN.JK: BERHASIL (4 baris data)
[10/915] DSFI.JK: BERHASIL (4 baris data)
[11/915] DEWI.JK: BERHASIL (4 baris data)
[12/915] ISEA.JK: BERHASIL (4 baris data)
[13/915] DPUM.JK: BERHASIL (4 baris data)
[14/915] ENZO.JK: BERHASIL (4 baris data)
[15/915] NEST.JK: BERHASIL (4 baris data)
[16/915] UDNG.JK: BERHASIL (4 baris data)
[17/915] WMPP.JK: BERHASIL (4 baris data)
[18/915] AGAR.JK: BERHASIL (4 baris data)
[19/915] AMMS.JK: BERHASIL (4 baris data)
[20/915] CRAB.JK: BERHASIL (4 baris data)
[21/915] SIPD.JK: BERHASIL (4 baris data)
[22/915] EPMT.JK: BERHASIL (4 baris data)
[23/915] SDPC.JK: BERHASIL (4 baris data)
[24/915] DAYA.J

$MAGP.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['MAGP.JK']: possibly delisted; no price data found  (period=5d)


[120/915] MAGP.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance


$GOLL.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['GOLL.JK']: possibly delisted; no price data found  (period=5d)


[121/915] GOLL.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[122/915] ULTJ.JK: BERHASIL (4 baris data)
[123/915] CMRY.JK: BERHASIL (4 baris data)
[124/915] CAMP.JK: BERHASIL (4 baris data)
[125/915] KEJU.JK: BERHASIL (4 baris data)
[126/915] TGUK.JK: BERHASIL (4 baris data)
[127/915] AMRT.JK: BERHASIL (4 baris data)
[128/915] MLPL.JK: BERHASIL (4 baris data)
[129/915] MPPA.JK: BERHASIL (4 baris data)
[130/915] MIDI.JK: BERHASIL (4 baris data)
[131/915] LAPD.JK: BERHASIL (4 baris data)
[132/915] RANC.JK: BERHASIL (4 baris data)
[133/915] HERO.JK: BERHASIL (4 baris data)
[134/915] JECX.JK: BERHASIL (4 baris data)
[135/915] DKHH.JK: BERHASIL (4 baris data)
[136/915] MIKA.JK: BERHASIL (4 baris data)
[137/915] SILO.JK: BERHASIL (4 baris data)
[138/915] HEAL.JK: BERHASIL (4 baris data)
[139/915] PRIM.JK: BERHASIL (4 baris data)
[140/915] SAME.JK: BERHASIL (4 baris data)
[141/915] PRDA.JK: BERHASIL (4 baris data)
[142/915] DGNS.JK: BERHASIL (4 baris data)
[143/915] CARE.JK:

...

[851/915] TECH.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance


$ENVY.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['ENVY.JK']: possibly delisted; no price data found  (period=5d)


[852/915] ENVY.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance


$LMAS.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['LMAS.JK']: possibly delisted; no price data found  (period=5d)


[853/915] LMAS.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[854/915] EDGE.JK: BERHASIL (4 baris data)
[855/915] PTSN.JK: BERHASIL (4 baris data)
[856/915] MSTI.JK: BERHASIL (4 baris data)
[857/915] GOTO.JK: BERHASIL (4 baris data)
[858/915] WIFI.JK: BERHASIL (4 baris data)
[859/915] EMTK.JK: BERHASIL (4 baris data)
[860/915] BUKA.JK: BERHASIL (4 baris data)
[861/915] KIOS.JK: BERHASIL (4 baris data)
[862/915] TOSK.JK: BERHASIL (4 baris data)
[863/915] UVCR.JK: BERHASIL (4 baris data)
[864/915] JATI.JK: BERHASIL (4 baris data)
[865/915] DIVA.JK: BERHASIL (4 baris data)
[866/915] HDIT.JK: BERHASIL (4 baris data)
[867/915] MPIX.JK: BERHASIL (4 baris data)
[868/915] KREN.JK: BERHASIL (4 baris data)
[869/915] MCAS.JK: BERHASIL (4 baris data)
[870/915] TFAS.JK: BERHASIL (4 baris data)
[871/915] AWAN.JK: BERHASIL (4 baris data)
[872/915] BELI.JK: BERHASIL (4 baris data)
[873/915] CASH.JK: BERHASIL (4 baris data)
[874/915] NFCX.JK: BERHASIL (4 baris data)
[875/915] PGJO.JK: